# Week 70, Day 4: Practice exam

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w070d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


*Interactive figure on the website: The timer keeps running if you reload. It doesn't stop you working when time is up..*

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: P1: Chunks

Write a generator `chunked(iterable, size)` yielding lists of `size` consecutive items (the last may be shorter, and no empty list is yielded). It must be lazy, working on infinite iterators, and must raise `ValueError` for a `size` below 1 **when called**, not when first iterated (so wrap a generator in an ordinary function).

In [ ]:
def chunked(iterable, size):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_chunked():\n    \"lazy chunks\"\n    import itertools\n    assert list(chunked(range(7), 3)) == [[0, 1, 2], [3, 4, 5], [6]] and list(chunked([], 2)) == []\n    assert list(itertools.islice(chunked(itertools.count(), 2), 2)) == [[0, 1], [2, 3]]\n    try:\n        chunked([1], 0)\n        assert False\n    except ValueError:\n        pass")

### Exercise: P2: A rate limiter

Write `limit_rate(calls, per, clock)`: a decorator factory. The decorated function may run at most `calls` times in any window of `per` seconds, measured with `clock()` (a function returning the current time, passed in so it can be tested). A call that would exceed the limit doesn't run, returns `None`, and adds one to the wrapper's `dropped` attribute. Keep the function's name. (A call at time `t` counts against later calls made before `t + per`.)

In [ ]:
import functools
from collections import deque

def limit_rate(calls, per, clock):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_limit():\n    \"a sliding window\"\n    now = [0]\n    @limit_rate(2, 10, lambda: now[0])\n    def ping(x):\n        return x * 2\n    results = []\n    for t in [0, 1, 2, 9, 10, 11, 12]:\n        now[0] = t\n        results.append(ping(t))\n    assert results == [0, 2, None, None, 20, 22, None] and ping.dropped == 3 and ping.__name__ == \"ping\"")

### Exercise: P3: All or nothing

Write a context manager `atomic(obj)` (with `@contextmanager`): on entry it saves a **deep** copy of `obj.__dict__`; if the block raises, it restores the object's attributes to exactly the saved state (including removing attributes added in the block) and lets the exception propagate; if the block succeeds, the changes stay. It yields `obj`.

In [ ]:
import copy
from contextlib import contextmanager

@contextmanager
def atomic(obj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_atomic():\n    \"a transaction\"\n    class Cart:\n        def __init__(self):\n            self.items, self.total = [\"tea\"], 3\n    c = Cart()\n    with atomic(c) as same:\n        assert same is c\n        c.items.append(\"jam\")\n        c.total += 2\n    assert c.items == [\"tea\", \"jam\"] and c.total == 5\n    try:\n        with atomic(c):\n            c.items.append(\"cake\")\n            c.total = 99\n            c.note = \"temp\"\n            raise RuntimeError\n    except RuntimeError:\n        pass\n    assert c.items == [\"tea\", \"jam\"] and c.total == 5 and not hasattr(c, \"note\")")

### Exercise: P4: Units as descriptors

Write a descriptor `Scaled(factor)` that presents a value stored in the instance's `metres` attribute in another unit: reading `obj.km` returns `obj.metres / factor` and setting `obj.km = v` sets `obj.metres = v * factor` (with `km = Scaled(1000)`). Looked up on the class, it returns itself. Then a class `Distance(metres)` with `km = Scaled(1000)`, `cm = Scaled(0.01)` and `miles = Scaled(1609.344)`, ordering and equality by `metres` (use `functools.total_ordering`), and `__repr__` giving `Distance(1500m)` (metres rounded to a whole number).

In [ ]:
import functools

class Scaled:
    ___

class Distance:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_units():\n    \"descriptors that share storage\"\n    d = Distance(1500)\n    assert d.km == 1.5 and d.cm == 150000 and repr(d) == \"Distance(1500m)\"\n    d.km = 2\n    assert d.metres == 2000 and vars(d) == {\"metres\": 2000}\n    d.miles = 1\n    assert round(d.metres, 3) == 1609.344 and isinstance(Distance.km, Scaled)\n    assert Distance(5) < Distance(10) and Distance(10) >= Distance(10) and Distance(3) == Distance(3)\n    assert sorted([Distance(3), Distance(1), Distance(2)]) == [Distance(1), Distance(2), Distance(3)]")

### Exercise: P5: A worker pool, simulated

Simulate a pool of `workers` workers (numbered from 0) given jobs in order: each job goes to the worker that becomes free **first** (ties to the lowest number), and starts when that worker is free. Write `schedule(durations, workers)` returning `(plan, makespan)`: `plan` is a list of `(worker, start, end)`, one per job in the given order, and `makespan` is when the last job ends (0 for no jobs). Then `best_order(durations, workers)`: the makespan when the jobs are given longest first (a good greedy order).

In [ ]:
import heapq

def schedule(durations, workers):
    ___

def best_order(durations, workers):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pool():\n    \"first free worker\"\n    plan, end = schedule([4, 2, 3, 1], 2)\n    assert plan == [(0, 0, 4), (1, 0, 2), (1, 2, 5), (0, 4, 5)] and end == 5\n    assert schedule([], 3) == ([], 0) and schedule([5], 3) == ([(0, 0, 5)], 5)\n    assert schedule([1, 1, 1, 6], 3)[1] == 7 and best_order([1, 1, 1, 6], 3) == 6")

### Exercise: C: A lazy query builder

Write a class `Query(source)` for building lazy pipelines over any iterable. Each of these returns a **new** `Query` and leaves the original unchanged: `where(pred)` keeps items for which `pred(item)` is true; `select(func)` maps; `limit(n)` keeps at most the first `n` items and **stops reading** the source after that; `batch(n)` groups items into lists of `n` (the last may be shorter). Nothing is read from the source until the query is iterated. Iterating a `Query` (`__iter__`) runs the pipeline afresh each time (so a query over a list can be iterated many times), and records in the query's `read` attribute how many source items that run took. Also `count()` (the number of results), `first(default=None)` (the first result, reading as little as possible) and `to_list()`.

In [ ]:
import itertools

class Query:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_query_basics():\n    \"chaining\"\n    q = Query(range(10))\n    evens = q.where(lambda x: x % 2 == 0)\n    squares = evens.select(lambda x: x * x)\n    assert squares.to_list() == [0, 4, 16, 36, 64] and squares.to_list() == [0, 4, 16, 36, 64]\n    assert q.to_list() == list(range(10)) and evens.count() == 5\n    assert Query(\"abcde\").batch(2).to_list() == [[\"a\", \"b\"], [\"c\", \"d\"], [\"e\"]]\n    assert Query([]).first(\"none\") == \"none\" and Query([3, 4]).first() == 3\n\ndef test_query_lazy():\n    \"laziness and reading as little as possible\"\n    import itertools\n    pulled = []\n    def source():\n        for i in itertools.count():\n            pulled.append(i)\n            yield i\n    q = Query(source()).where(lambda x: x % 3 == 0).limit(3)\n    assert pulled == []\n    assert q.to_list() == [0, 3, 6] and q.read == 7 and pulled == list(range(7))\n    words = Query([\"kiwi\", \"fig\", \"banana\", \"apple\", \"plum\"])\n    first_long = words.where(lambda w: len(w) > 4)\n    assert first_long.first() == \"banana\" and first_long.read == 3\n    assert words.select(len).limit(2).batch(5).to_list() == [[4, 3]]")

## How to use this

Treat the first two hours as the real thing: notes closed, no tutor, no web, timer on. Run your code and press **Check** as often as you like. Unlike Friday, each problem has a hint and a model solution: **don't open them until the timer has finished**. Then go back through every problem and compare.

**The marks.**

| Part | What | Marks |
|---|---|---|
| A | 10 quiz questions | 10 (1 each) |
| B | 5 problems | 70 |
| C | one larger program | 30 |
| | **Total** | **110** |

Count full marks for a problem only when every check passes. 66 (60%) would be a pass, 88 (80%) a distinction.

## Part A: quiz

## Part B: five problems

## Part C: a larger program

## Afterwards

When the timer ends, add up your marks, then go back through each problem with its hint and solution. For every mark you dropped, write one line: the problem, the reason (**didn't know it**, **misread it**, **an edge case**, **ran out of time**) and the week to revise.

Which ideas did today need? P1 generators and the "check eagerly, iterate lazily" pattern (Week 59); P2 a decorator factory with closure state (Week 60); P3 a generator context manager with deep copies (Weeks 61 and 62); P4 descriptors that share storage, and `total_ordering` (Weeks 63 and 60); P5 a simulated worker pool with a heap (Week 66); C a lazy pipeline of generator stages built from immutable objects (Weeks 59 and 60).

If the score disappointed you, don't spend the time before the exam reading: redo the problems you dropped marks on, without looking, until they pass, then retry the review days' bug hunts and challenges.